# Comparación de Condiciones

El dataset escogido es sobre precios de casas en EEUU. Se propone la comparación de la variable pies cuadrados de terreno con precio de la casa.

In [ ]:
import os
from pathlib import Path
from functools import cache

import kagglehub
import pandas as pd


def get_cache_dir() -> Path:
    """
    Creates and returns the cache directory for kagglehub.
    """
    cache_dir = Path.cwd() / "data" / "raw"
    cache_dir.mkdir(parents=True, exist_ok=True)

    os.environ["KAGGLEHUB_CACHE"] = str(cache_dir)

    return cache_dir


@cache
def get_data() -> pd.DataFrame:
    """
    Downloads the dataset from Kaggle and loads it into a DataFrame.
    """

    print("Cache directory:", get_cache_dir())

    data_path = kagglehub.dataset_download(
        "debayank2024/house-price-prediction"
    )

    file_path = Path(data_path) / "modified_data.csv"

    df = pd.read_csv(file_path)

    print("Path to file:", file_path)

    return df

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from plotnine import *

df = get_data()

### División del dataset por tamaño de vivienda

Con el fin de poder realizar la prueba t, se decidió dividir las viviendas utilizando la mediana de la variable sqft_living como punto de división, categorizando entonces las viviendas en "casas pequeñas" y "casas grandes".

In [5]:
# Mediana de sqft_living
mediana_sqft = df['sqft_living'].median()

# División en dos dataframes: casas pequeñas y casas grandes
casas_pequenas = df[df['sqft_living'] < mediana_sqft].copy()
casas_grandes = df[df['sqft_living'] >= mediana_sqft].copy()

### Prueba de normalidad

Se utiliza la prueba de Shapiro-Wilk para verificar la normalidad de `price_per_sqft` en ambos grupos.

**H0:** Los datos siguen una distribución normal.  
**H1:** Los datos no siguen una distribución normal.


In [ ]:
from scipy import stats

variable = 'price_per_sqft'

stat_pequenas, p_pequenas = stats.shapiro(casas_pequenas[variable].dropna())
stat_grandes, p_grandes = stats.shapiro(casas_grandes[variable].dropna())

print(f"Casas pequeñas - W: {stat_pequenas:.4f}, p-valor: {p_pequenas:f}")
print(f"Casas grandes - W: {stat_grandes:.4f}, p-valor: {p_grandes:f}")


### Prueba de igualdad de varianzas (Brown-Forsythe)

Para verificar la hipótesis de igualdad de varianzas entre los grupos de casas pequeñas y casas 
grandes, se utilizó la prueba de Brown-Forsythe, modificación de la prueba de Levene que centra los grupos usando la mediana en vez de la media. La razón de utilizar esta prueba se debe a que en el EDA realizado anteriormente con el dataset se descubrieron multiples valores atípicos en la variable price_per_sqft. La hipótesis plantea que la varianza de ambos grupos son iguales.

In [ ]:
from scipy import stats

variable = 'price_per_sqft'

stat, p_value = stats.levene(casas_pequenas[variable], casas_grandes[variable], center='median')

print(f"Estadístico de Brown-Forsythe: {stat:.4f}")
print(f"p-valor: {p_value:f}")

alpha = 0.05
if p_value < alpha:
    print("Se rechaza H0: las varianzas son diferentes")
else:
    print("No se rechaza H0: no hay diferencia entre las varianzas")

Estadístico de Brown-Forsythe: 3.8913
Valor P: 0.048598
Se rechaza H0: las varianzas son diferentes


Con un p-valor de 0.048598 (< 0.05), se rechaza H0: las varianzas de price_per_sqft entre 
casas pequeñas y casas grandes son significativamente diferentes, aunque es 
significativo por un margen pequeño. Con esto hecho, se debe proseguir con una prueba t de Welch.

### Prueba t

Como la prueba de Brown-Forsythe indicó varianzas diferentes, se utiliza la prueba t de Welch.

**H0:** Las medias de `price_per_sqft` son iguales en ambos grupos.  
**H1:** Las medias de `price_per_sqft` son diferentes entre los grupos.


In [ ]:
stat_t, p_t = stats.ttest_ind(
    casas_pequenas[variable],
    casas_grandes[variable],
    equal_var=False,
    nan_policy='omit'
)

print(f"Estadístico t: {stat_t:.4f}")
print(f"p-valor: {p_t:f}")
